# Assignment 1: Myanmar Part-of-Speech Tagging using CRF

**Name:** Myint Thu Soe  
**Dataset:** myPOS Version 3.0  
**Model:** Conditional Random Field  
**Due Date:** 31 July 2026

In [ ]:
%pip install -q sklearn-crfsuite scikit-learn tabulate

In [ ]:
import os
from pathlib import Path
from collections import Counter

import sklearn_crfsuite
from sklearn_crfsuite import metrics
from sklearn.model_selection import train_test_split
from tabulate import tabulate

print("All required libraries imported successfully.")

In [ ]:
from pathlib import Path

possible_files = [
    Path("mypos-ver.3.0.shuf.nopipe.txt"),
    Path("mypos-ver.3.0.shuf.txt")
]

DATASET_PATH = None

for file_path in possible_files:
    if file_path.exists():
        DATASET_PATH = file_path
        break

if DATASET_PATH is None:
    print("Dataset file was not found.")
    print("Files currently available:")
    
    for file_path in Path.cwd().iterdir():
        print("-", file_path.name)
else:
    print("Dataset found successfully.")
    print("Using dataset:", DATASET_PATH.name)

In [ ]:
if DATASET_PATH is None:
    raise FileNotFoundError(
        "Please add the POS-tagged myPOS dataset to this folder."
    )

with open(DATASET_PATH, "r", encoding="utf-8") as file:
    for line_number in range(5):
        line = file.readline()

        if not line:
            break

        print(f"Line {line_number + 1}:")
        print(line.strip())
        print()

In [ ]:
def load_mypos_data(file_path):
    sentences = []
    invalid_items = 0

    with open(file_path, "r", encoding="utf-8") as file:
        for line in file:
            line = line.strip()

            if not line:
                continue

            # Official corpus compound-word pipe ကို token separator ပြောင်းရန်
            line = line.replace("|", " ")

            sentence = []

            for item in line.split():
                if "/" not in item:
                    invalid_items += 1
                    continue

                word, tag = item.rsplit("/", 1)

                word = word.strip()
                tag = tag.strip()

                if word and tag:
                    sentence.append((word, tag))
                else:
                    invalid_items += 1

            if sentence:
                sentences.append(sentence)

    return sentences, invalid_items


dataset, invalid_items = load_mypos_data(DATASET_PATH)

print(f"Total sentences loaded: {len(dataset):,}")
print(f"Invalid items skipped: {invalid_items:,}")

if dataset:
    print("\nFirst parsed sentence:")
    print(dataset[0])

In [ ]:
if not dataset:
    raise ValueError("No valid POS-tagged sentences were loaded.")

all_tags = [
    tag
    for sentence in dataset
    for word, tag in sentence
]

tag_counts = Counter(all_tags)

print("Number of sentences:", len(dataset))
print("Number of tagged words:", len(all_tags))
print("Number of unique POS tags:", len(tag_counts))

print("\nPOS tag distribution:")

for tag, count in sorted(tag_counts.items()):
    print(f"{tag:>6} : {count:,}")

In [ ]:
def word2features(sentence, index):
    word = sentence[index][0]

    features = {
        "bias": 1.0,
        "word": word,
        "word.length": len(word),
        "word.prefix1": word[:1],
        "word.prefix2": word[:2],
        "word.suffix1": word[-1:],
        "word.suffix2": word[-2:],
        "word.isdigit": word.isdigit()
    }

    # Previous word features
    if index > 0:
        previous_word = sentence[index - 1][0]

        features.update({
            "-1:word": previous_word,
            "-1:word.length": len(previous_word),
            "-1:word.prefix1": previous_word[:1],
            "-1:word.suffix1": previous_word[-1:]
        })
    else:
        features["BOS"] = True

    # Next word features
    if index < len(sentence) - 1:
        next_word = sentence[index + 1][0]

        features.update({
            "+1:word": next_word,
            "+1:word.length": len(next_word),
            "+1:word.prefix1": next_word[:1],
            "+1:word.suffix1": next_word[-1:]
        })
    else:
        features["EOS"] = True

    return features


def sent2features(sentence):
    return [
        word2features(sentence, index)
        for index in range(len(sentence))
    ]


def sent2labels(sentence):
    return [
        tag
        for word, tag in sentence
    ]


def sent2tokens(sentence):
    return [
        word
        for word, tag in sentence
    ]

In [ ]:
print("Tokens:")
print(sent2tokens(dataset[0]))

print("\nLabels:")
print(sent2labels(dataset[0]))

print("\nFeatures of first word:")
print(sent2features(dataset[0])[0])

In [ ]:
train_sentences, test_sentences = train_test_split(
    dataset,
    test_size=0.20,
    random_state=42
)

X_train = [
    sent2features(sentence)
    for sentence in train_sentences
]

y_train = [
    sent2labels(sentence)
    for sentence in train_sentences
]

X_test = [
    sent2features(sentence)
    for sentence in test_sentences
]

y_test = [
    sent2labels(sentence)
    for sentence in test_sentences
]

print(f"Training sentences: {len(X_train):,}")
print(f"Testing sentences:  {len(X_test):,}")

In [ ]:
crf_model = sklearn_crfsuite.CRF(
    algorithm="lbfgs",
    c1=0.1,
    c2=0.1,
    max_iterations=100,
    all_possible_transitions=True
)

print("Training CRF POS tagger...")
print("This process may take several minutes.")

crf_model.fit(X_train, y_train)

print("CRF model training completed successfully.")

In [ ]:
print("Predicting POS tags for testing data...")

y_pred = crf_model.predict(X_test)

labels = sorted(crf_model.classes_)

accuracy = metrics.flat_accuracy_score(
    y_test,
    y_pred
)

precision = metrics.flat_precision_score(
    y_test,
    y_pred,
    average="weighted",
    labels=labels,
    zero_division=0
)

recall = metrics.flat_recall_score(
    y_test,
    y_pred,
    average="weighted",
    labels=labels,
    zero_division=0
)

f1_score = metrics.flat_f1_score(
    y_test,
    y_pred,
    average="weighted",
    labels=labels,
    zero_division=0
)

print("\n=== Model Evaluation Results ===")
print(f"Accuracy:           {accuracy * 100:.2f}%")
print(f"Weighted Precision: {precision * 100:.2f}%")
print(f"Weighted Recall:    {recall * 100:.2f}%")
print(f"Weighted F1-score:  {f1_score * 100:.2f}%")

In [ ]:
print("=== Detailed Classification Report ===")

report = metrics.flat_classification_report(
    y_test,
    y_pred,
    labels=labels,
    digits=4,
    zero_division=0
)

print(report)

In [ ]:
def tag_custom_sentence(words, model):
    dummy_sentence = [
        (word, "UNKNOWN")
        for word in words
    ]

    features = sent2features(dummy_sentence)
    predicted_tags = model.predict_single(features)

    return list(zip(words, predicted_tags))

In [ ]:
sample_words = [
    "ကျောင်းသား",
    "များ",
    "သည်",
    "စာကြည့်တိုက်",
    "တွင်",
    "စာဖတ်",
    "နေ",
    "ကြ",
    "သည်",
    "။"
]

prediction_result = tag_custom_sentence(
    sample_words,
    crf_model
)

print(
    tabulate(
        prediction_result,
        headers=["Word", "Predicted POS Tag"],
        tablefmt="grid"
    )
)

In [ ]:
error_rows = []

for sentence, true_tags, predicted_tags in zip(
    test_sentences,
    y_test,
    y_pred
):
    words = sent2tokens(sentence)

    for word, true_tag, predicted_tag in zip(
        words,
        true_tags,
        predicted_tags
    ):
        if true_tag != predicted_tag:
            error_rows.append([
                word,
                true_tag,
                predicted_tag
            ])

print("Total incorrectly predicted words:", len(error_rows))

print(
    tabulate(
        error_rows[:20],
        headers=[
            "Word",
            "True Tag",
            "Predicted Tag"
        ],
        tablefmt="grid"
    )
)

## Result Analysis

The Myanmar POS tagger was developed using the myPOS
Version 3.0 dataset and a Conditional Random Field model.

The dataset was divided into 80% training data and 20%
testing data. Word identity, word length, prefixes, suffixes,
previous words, next words, and sentence-boundary features
were used for CRF training.

The trained model achieved an accuracy of 111513% and a
weighted F1-score of 0.9599%.

The model performed well for frequently occurring POS tags,
including nouns, verbs, particles, post-positional markers,
and punctuation. Some errors were observed among tags with
similar grammatical functions or fewer training examples.

A manually segmented Myanmar sentence was also tested, and
the model successfully predicted a POS tag for each word.